In [96]:
# import Library
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score
from sklearn.linear_model import LogisticRegression


In [97]:
# load the data
df=pd.read_csv('online_retail.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [98]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [99]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [100]:
df.dropna(subset='CustomerID',inplace=True)

In [101]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  object 
 1   StockCode    406829 non-null  object 
 2   Description  406829 non-null  object 
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  object 
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      406829 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 27.9+ MB


In [103]:
# Convert the str datatype to Dateandtime:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

# Setting maximum date
Prefered_date =pd.to_datetime('2011-12-08')


In [104]:
df['since_date']=(Prefered_date - df['InvoiceDate'] ).dt.days

df["Price"]=df["Quantity"] * df['UnitPrice']


In [105]:
# RFM AGGREGATION
Recency=df.groupby('CustomerID')['since_date'].max().reset_index()
Frequency=df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
Monetary = df.groupby('CustomerID')["Price"].sum().reset_index()

In [106]:
# Merge the data together by Customer ID
rfm=Recency.merge(Frequency,on='CustomerID').merge(Monetary,on='CustomerID')
rfm.columns=('CustomerID' ,	'Recency' ,	'Frequency' ,'Monetary')
rfm

,CustomerID,Recency,Frequency,Monetary
0,12346.0,323,2,0.00
1,12347.0,365,7,4310.00
2,12348.0,356,4,1797.24
3,12349.0,16,1,1757.55
4,12350.0,308,1,334.40
...,...,...,...,...
4367,18280.0,275,1,180.60
4368,18281.0,178,1,80.82
4369,18282.0,124,3,176.60
4370,18283.0,335,16,2094.88


In [107]:
# RULE-BASED RFM SEGMENTATION
rfm["R"]=pd.qcut(rfm["Recency"],q=5,labels=[5,4,3,2,1])
rfm["F"]=pd.qcut(rfm["Frequency"].rank(method='first'),q=5,labels=[5,4,3,2,1])
rfm["M"]=pd.qcut(rfm["Monetary"],q=5,labels=[5,4,3,2,1])

rfm["rfm_segmentaion"] = rfm["R"].astype(str)+rfm["F"].astype(str)+rfm["M"].astype(str)
rfm


,CustomerID,Recency,Frequency,Monetary,R,F,M,rfm_segmentaion
0,12346.0,323,2,0.00,2,4,5,245
1,12347.0,365,7,4310.00,1,2,1,121
2,12348.0,356,4,1797.24,1,3,2,132
3,12349.0,16,1,1757.55,5,5,2,552
4,12350.0,308,1,334.40,2,5,4,254
...,...,...,...,...,...,...,...,...
4367,18280.0,275,1,180.60,3,4,5,345
4368,18281.0,178,1,80.82,4,4,5,445
4369,18282.0,124,3,176.60,4,3,5,435
4370,18283.0,335,16,2094.88,2,1,1,211


In [108]:
# Assigning the Chuns
rfm['Chuns'] = (rfm['Frequency'] < 3).astype(int)

# Check distribution
print(rfm['Chuns'].value_counts())

Chuns
0    2242
1    2130
Name: count, dtype: int64


In [109]:
# Define Dependent and Independent (X and Y)
x=rfm[['Frequency','Monetary']]
y=rfm['Chuns']

In [110]:
# Train Test Split The data 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [111]:
# Standardization
scale=StandardScaler()
scaled_train=scale.fit_transform(x_train)
scaled_test=scale.transform(x_test)

In [112]:
# THE MODEL SELECTION LOOP
models={
    'Logistic Regression':LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results=[]

In [113]:
for name,model in models.items():
    # Training on correct scaled training split
    model.fit(scaled_train,y_train)
    prediction=model.predict(scaled_test)

    Pre = precision_score(y_test,prediction)
    f1 = f1_score(y_test,prediction)
    Recall=recall_score(y_test,prediction)
    Acc=accuracy_score(y_test,prediction)

    results.append({"name":name,'precision':Pre,'F!':f1,'Recall':Recall,"accuracy":Acc})




In [114]:
# score Board
score_board=pd.DataFrame(results)
print(score_board)

                  name  precision       F!    Recall  accuracy
0  Logistic Regression        1.0  0.99883  0.997664  0.998857
1        Random Forest        1.0  1.00000  1.000000  1.000000
2    Gradient Boosting        1.0  1.00000  1.000000  1.000000


In [115]:
# SIMPLE CHURN PROBABILITY BASED ON RECENCY
rfm["chuns_prob"]=(rfm['Recency']/365).clip(0.01,1.0)

# THE AVERAGE PRICE PER TRANSACTION
rfm["purchase_price"]=rfm["Monetary"]/rfm['Frequency']

# CLV FORMULA
rfm['CLV']=(rfm["purchase_price"] * rfm['Frequency'])/rfm["chuns_prob"]

In [116]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R,F,M,rfm_segmentaion,Chuns,chuns_prob,purchase_price,CLV
0,12346.0,323,2,0.00,2,4,5,245,1,0.884932,0.000000,0.000000
1,12347.0,365,7,4310.00,1,2,1,121,0,1.000000,615.714286,4310.000000
2,12348.0,356,4,1797.24,1,3,2,132,0,0.975342,449.310000,1842.675843
3,12349.0,16,1,1757.55,5,5,2,552,1,0.043836,1757.550000,40094.109375
4,12350.0,308,1,334.40,2,5,4,254,1,0.843836,334.400000,396.285714


In [117]:
import pickle

best_model = models['Logistic Regression']

with open('churn_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scale, f)

print("✅ Model saved!")

✅ Model saved!
